In [1]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import datetime

print("Imports loaded.")

Imports loaded.


In [2]:
ee.Authenticate()

True

In [3]:
ee.Initialize(project="unique-spirit-494108-g2")
print("Earth Engine initialized OK.")

Earth Engine initialized OK.


Question (4 pts): Throughout this lab, we will be comparing SAR GRD level data to compare the same AOI over different periods of time. Why is it important to keep the orbit direction consistent when comparing scenes over time?

ANSWER: Keeping the orbit direction consistent is important because SAR images change depending on whether the satellite is in ascending or descending pass, which affects viewing geometry. If orbit direction differs between scenes, apparent changes may come from imaging geometry (e.g., layover, shadow, backscatter differences) rather than real changes on the ground.

In [4]:
# Area of interest: Kraków and surrounding urban/peri-urban area
aoi = ee.Geometry.Rectangle([19.85, 49.95, 20.20, 50.14])

# Analysis settings
ORBIT_PASS = "ASCENDING"
MODE = "IW"
RESOLUTION = 10

# Year range used later in the notebook
years = list(range(2015, 2025))

m = geemap.Map()
m.centerObject(aoi, 10)
m.addLayer(aoi, {"color": "red"}, "AOI")
display(m)

Map(center=[50.04506888629718, 20.024999999999867], controls=(WidgetControl(options=['position', 'transparent_…

In [7]:
def get_s1_collection(start_date, end_date, region=aoi,
                      orbit_pass=ORBIT_PASS,
                      mode=MODE,
                      resolution=RESOLUTION):
    """
    Return a filtered Sentinel-1 GRD ImageCollection.
    """
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD")                     # loads all Sentinel images
        .filterBounds(region)                                       # only from Kraków
        #YOUR CODE HERE (10 pts)
        .filterDate(start_date, end_date)                           # specific time range
        .filter(ee.Filter.eq('instrumentMode', mode))               # specific radar mode
        .filter(ee.Filter.eq('orbitProperties_pass', orbit_pass))   # satelite direction
        .filter(ee.Filter.eq('resolution_meters', resolution))      # specific resolution
    )

    return collection


def get_monthly_median(year, month=8, region=aoi):
    """
    Return the median Sentinel-1 image for a given month and year.
    By default, this is August because it reduces seasonal variation
    and avoids snow-related complications.
    """
    start = f"{year}-{month:02d}-01"                        # first day of the month
    if month == 12:                     
        end = f"{year + 1}-01-01"                           # for december goes to january 01 next year
    else:
        end = f"{year}-{month + 1:02d}-01"                  # for other monts goes for first of the next month

    #YOUR CODE HERE (10 pts). IN SECTION 4, YOU MANUALLY GOT THE COLLECTION AND SET THE MEDIAN.
    #SINCE WE INTEND TO DO THIS MULTIPLE TIMES, LETS PUT THIS IN A FUNCTION

    collection = get_s1_collection(start, end, region)      # filtered Sentiel collection

    image = collection.median()                             # median image
    
    return image.set({                                      # ads month, year and count of images from Sentiel to the photo
        "year": year,
        "month": month,
        "scene_count": collection.size()
    })


def show_scene_metadata(collection):
    """
    Print selected metadata for scenes in a collection.
    Useful for discussing repeat intervals and duplicate dates.
    """
    #YOUR CODE HERE (10 pts). Fill in this function with additional metadata required to answer any questions you may have
    times = collection.aggregate_array("system:time_start").getInfo()                               # pulls a timestamps in milliseconds
    passes = collection.aggregate_array("orbitProperties_pass").getInfo()                           # ascending or descending
    platforms = collection.aggregate_array("platform_number").getInfo()                             # Sentinel-1A or 1B
    orbit_ralative = collection.aggregate_array("relativeOrbitNumber_start").getInfo()              # 
    slice_number = collection.aggregate_array("sliceNumber").getInfo()
    total_sloices = collection.aggregate_array("totalSlices").getInfo()
    models = collection.aggregate_array("instrumentMode").getInfo()
    polarization = collection.aggregate_array("transmitterReceiverPolarisation").getInfo()          # polarization for every image (VV or VH)
    resolution = collection.aggregate_array("resolution_meters").getInfo()
    angles = collection.aggregate_array("meanIncidenceAngle").getInfo()                             # look angle for backscatter values
    orbit_number = collection.aggregate_array("orbitNumber_start").getInfo()                        # exact satellite orbit number 

    # combines elements from each list
    for t, p, sat, pol, ang, orb_n in zip(times, passes, platforms, orbit_ralative, slice_number, total_sloices, models, polarization, resolution, angles, orbit_number):
        dt = datetime.datetime.utcfromtimestamp(t / 1000)                                           # converts to seconds then to normal date
        print(f"{dt.date()} | {p} | S1{sat} | Orbit {orb_n} | Angle {ang:.2f} | {pol} ")              # 2026-04-22 | ASCENDING | S1A | Orbit 45 | Angle 34.55 | VH


def fixed_histogram(image, region, band, hist_min, hist_max, n_bins=100, scale=10):
    """
    Compute a fixed histogram for one band of an Earth Engine image.
    Returns bin centers, counts, and bin width.
    """
    hist = image.select(band).reduceRegion(
        reducer=ee.Reducer.fixedHistogram(hist_min, hist_max, n_bins),
        geometry=region,
        scale=scale,
        maxPixels=1e9,
        bestEffort=True
    ).get(band)

    hist = ee.List(hist).getInfo()
    bin_edges = [row[0] for row in hist]
    counts = [row[1] for row in hist]
    bin_width = (hist_max - hist_min) / n_bins
    bin_centers = [edge + bin_width / 2 for edge in bin_edges]

    return bin_centers, counts, bin_width


def normalize_counts(counts):
    total = sum(counts)
    if total == 0:
        return counts
    return [c / total for c in counts]


def region_stats(image, region, band):
    """
    Return mean, median, std dev, and max for a selected band.
    """
    stats = image.select(band).reduceRegion(
        reducer=(
            ee.Reducer.mean()
            .combine(ee.Reducer.median(), sharedInputs=True)
            .combine(ee.Reducer.stdDev(), sharedInputs=True)
            .combine(ee.Reducer.max(), sharedInputs=True)
        ),
        geometry=region,
        scale=10,
        maxPixels=1e9,
        bestEffort=True
    ).getInfo()

    return {
        "mean": stats.get(f"{band}_mean"),
        "median": stats.get(f"{band}_median"),
        "stdDev": stats.get(f"{band}_stdDev"),
        "max": stats.get(f"{band}_max"),
    }

print("Helper functions defined.")

Helper functions defined.


In [8]:
# Visualization presets
vv_vis = {"min": -20, "max": 0}
vh_vis = {"min": -28, "max": -5}
#YOU WILL NEED TO WRITE THE get_s1_collection FUNCTION TO GET THIS TO WORK. ONCE YOU HAVE
#THE FUNCTION WORKING, PLACE IT IN THE FUNCTIONS CODE BLOCK IN SECTION 3.
jan_2024 = get_s1_collection("2024-01-01", "2024-02-01", region=aoi)

#YOU WILL NEED TO ADAPT THE show_scene_metadata() FUNCTION TO ANSWER THE QUESTION BELOW
print("Image count:", jan_2024.size().getInfo())
show_scene_metadata(jan_2024)

jan_2024_img = jan_2024.median().clip(aoi)

m = geemap.Map()
m.centerObject(aoi, 10)

#If we were plotting just one img, you would simply use the line below. A split_map allows
#two scenes to be compared via a slider bar.
#m.addLayer(jan_2024_img.select("VV"), vv_vis, "VV January 2024")

m.split_map(
    left_layer=geemap.ee_tile_layer(jan_2024_img.select("VV"), vv_vis, "VV January 2024"),
    right_layer=geemap.ee_tile_layer(jan_2024_img.select("VH"), vh_vis, "VH January 2024")
)
display(m)

Image count: 10


Map(center=[50.04506888629718, 20.024999999999867], controls=(ZoomControl(options=['position', 'zoom_in_text',…

QUESTION (4 pts): We know Sentinel-1 has an approximately 6-day repeat time. However, from the metadata presented above, there are 10 images that GEE grabbed, two for each acquisition day. Why is that? Do not guess, look into the metadata for those scenes to get your answer. You will need to modify the metadata function in the helper functions code block above. Please also adjust the print function to print the metadata field you thinks supports your answer

Some common metadata fields are: time_start, platform_number, orbitProperties_pass, relativeOrbitNumber_start, sliceNumber, totalSlices, instrumentMode, transmitterReceiverPolarisation, resolution_meters, and orbitNumber_start.

ANSWER: The reason there are 10 images (2 per acquisition day) is that Sentinel-1 typically captures the same area twice per day on different orbital passes—once in an ascending pass and once in a descending pass. This is directly visible in the metadata field orbitProperties_pass, which shows two distinct values (ASCENDING and DESCENDING) for the same date, confirming the duplication is due to orbit direction rather than additional revisit days.